# Tier-2 (full architecture): genome-wide pretraining → CRISPR fine-tune

**What broke the first attempt.** The reduced Tier-2 trained the auxiliary 311-TF head on only ~4,000 CRISPR elements — so it never solved the data-starvation problem and landed at AUPRC **0.488** (below epigenetics-only 0.519).

**The real architecture, now implemented:**
1. **Pretrain** the shared conv encoder on **~100k genome-wide ENCODE cCREs** to predict which of **311 TFs** bind each locus (dense: ~31M labels). This forces the encoder to learn the TF motif grammar *from millions of loci*, not 4,000.
2. **Fine-tune** the sparse CRISPR head (569 positives) on top of the pretrained encoder.

The notebook runs fine-tune **with vs without** pretraining, so the pretraining benefit is isolated. Baselines (same chromosome-held-out protocol): distance **0.393**, epigenetics-only GBM **0.519**, TF-identity GBM **0.608**. All data (cCRE sequences+labels, CRISPR features) is committed to the repo. **Set Runtime → GPU.**

In [ ]:
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU  <-- Runtime > Change runtime type > GPU')

In [ ]:
# Clone the repo (all data + code committed). Private repo -> set TOKEN.
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
TOKEN  = ''
OWNER, REPO = 'nikku03', 'cell'
url = f'https://{TOKEN + "@" if TOKEN else ""}github.com/{OWNER}/{REPO}.git'
import os
if not os.path.isdir(REPO):
    !git clone --depth 1 --branch $BRANCH $url
%cd $REPO
!pip -q install xgboost
import sys; sys.path.insert(0, 'colab')
for f in ['outputs/orphan/invivo/element_seqs.json','outputs/orphan/invivo/compendium_tf.json','outputs/orphan/crispr_features_compendium.csv','outputs/orphan/invivo/pretrain/pretrain_seqs.txt.gz','outputs/orphan/invivo/pretrain/pretrain_labels.npz']:
    print(('OK ' if os.path.exists(f) else 'MISSING '), f)

## 1. Tier-1 GBM baselines (same chromosome-held-out protocol)

In [ ]:
import crispr_gate as g
I = g.identity_test()
for k in ['distance','epigenetics','epi+TF_identity_311','shuffled_control']:
    print(f'  {k:22s} AUPRC {I[k]["auprc"]:.3f}   seeds {I[k]["seeds"]}')

## 2. The full Tier-2: pretrain on cCREs → fine-tune (with vs without)
Pretrains the encoder on ~100k cCREs (prints aux TF-binding AUC per epoch), then fine-tunes the CRISPR head from-scratch and from-pretrained, chromosome-held-out, + a label-shuffle control. On a Colab GPU this is a few minutes.

In [ ]:
import seq_model as m
out = m.main_full()

## 3. Read the result

| comparison | meaning |
|---|---|
| pretrained > scratch (+0.03) | **genome-wide pretraining works** — the encoder transferred TF grammar to the sparse task |
| pretrained > 0.519 | beats epigenetics-only → learned regulation signal from sequence, ChIP-free |
| pretrained ≥ 0.608 | matches/beats the measured-ChIP GBM without ChIP — strong adopt |
| pretrained ≈ scratch | pretraining didn't transfer; ship the GBM |
| shuffle control ≈ 0.055 | confirms no leakage (must hold) |

The `main_full()` verdict line states the call. Whatever it lands on is the honest answer — this is now the architecture *as designed*, so the result is a fair test of it.